# **Pràctica**

## **1. Objectius**

L’objectiu principal d’aquesta pràctica és explorar l’aprenentatge no supervisat mitjançant l’entrenament d’un **autoencoder** en una **tasca de pretext**, i posteriorment aplicar el coneixement adquirit en un escenari de **transfer learning**.

En concret, es proposa entrenar un autoencoder que aprengui a **reconstruir la imatge original a partir d’una versió rotada** de la mateixa. Un cop entrenat, l’*encoder* resultant s’emprarà com a extractor de característiques per a una tasca de classificació d’imatges.


## **2. Conjunt de dades**

Per a la pràctica s’utilitzarà el **conjunt de dades Caltech101**, que conté imatges de 101 categories d’objectes naturals i artificials, amb una mida i resolució variables.
Les imatges s’hauran de:
- Redimensionar a una mida uniforme (p. ex. 128×128 píxels).
- Normalitzar (valors de píxel entre 0 i 1).
- Generar-ne versions rotades per construir el conjunt d’entrenament de la tasca de pretext.

Podeu descarregar-lo des d'aquest Kaggle: [enllaç](https://www.kaggle.com/datasets/imbikramsaha/caltech-101)


## **3. Recomacions**

1. Emprar quatre convolucions tant a l'encoder com al decoder.
2. Entrenar 100 èpoques mínim per la tasca de pretext.
3. Provar-ho primer amb 5 de les 100 classes.
4. **No empreu ChatGPT**, si pot ser.

### Descargar y preparar el dataset Caltech101

El error `FileNotFoundError` se produce porque el conjunto de datos no se encuentra en la ruta especificada en el entorno de Colab. Para solucionarlo, debemos descargar el conjunto de datos Caltech101 de Kaggle y colocarlo en la estructura de directorios esperada (`dataset/caltech-101`).

**Pasos para descargar el dataset:**

1.  **Obtener tu Kaggle API Key:**
    *   Ve a [Kaggle](https://www.kaggle.com/).
    *   Haz clic en tu foto de perfil en la esquina superior derecha y selecciona 'Your Profile'.
    *   Haz clic en 'Account' y desplázate hacia abajo hasta la sección 'API'.
    *   Haz clic en 'Create New API Token'. Esto descargará un archivo `kaggle.json`.
2.  **Subir `kaggle.json` a Colab:**
    *   En Colab, haz clic en el icono de la carpeta (Files) en el panel lateral izquierdo.
    *   Haz clic en el icono de 'Upload to session storage' (un archivo con una flecha hacia arriba).
    *   Sube el archivo `kaggle.json` que descargaste.
3.  **Ejecutar las siguientes celdas de código** para instalar la librería de Kaggle, configurar la API key y descargar el dataset.

In [14]:
# Instalar la librería de Kaggle
!pip install kaggle

# Descargar el dataset de Kaggle
# El enlace proporcionado en el cuaderno es: https://www.kaggle.com/datasets/imbikramsaha/caltech-101
!kaggle datasets download -d imbikramsaha/caltech-101

# Descomprimir el dataset
#!unzip caltech-101.zip -d dataset/

#!ls dataset


Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [11]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import glob
import pandas as pd
import numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import xml.etree.ElementTree as ET

# Tamaño de las imagenes y del batch size
IMAGE_SIZE = 128
BATCH_SIZE = 32

# Transoformador de las imagenes
transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                                transforms.ToTensor()])

# Dataset de caltech-101
dataset = datasets.ImageFolder(root="dataset/caltech-101", transform=transform)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/caltech-101'

In [ ]:
import random
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset

class RotatedDataset(Dataset):

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        imagen, _ = self.base_dataset[idx]
        angle = random.uniform(-45,45)
        rotated = TF.rotate(imagen, angle)
        return rotated, imagen

In [ ]:
# Divisio entre entrenament i validacio

pretext_dataset = RotatedDataset(dataset)
train_size = int(0.8 * len(pretext_dataset))
test_size = len(pretext_dataset) - train_size
train_ds, test_ds = random_split(pretext_dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

Autoencoder

In [ ]:
# AUTOENCODER

import torch
import torch.nn as nn

# Encoder
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),  # 128→64
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # 64→32
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # 32→16
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # 16→8
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

    def forward(self, x):
        return self.encoder(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # 8→16
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # 16→32
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),    # 32→64
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),     # 64→128
            nn.BatchNorm1d(3),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.decoder(x)

# Autoencoder que combina l'encoder i decoder
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

Confuguram la GPU si n'hiha

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Autoencoder().to(device)

criterion = nn.MSELoss()  # l’objectiu és reconstruir la imatge → MSE
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


Entrenament

In [ ]:
from tqdm import tqdm

EPOCHS = 100  # com a mínim 100 segons l’enunciat, però pots començar amb 10 per provar

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for x_rot, x_orig in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x_rot, x_orig = x_rot.to(device), x_orig.to(device)
        optimizer.zero_grad()
        outputs = model(x_rot)
        loss = criterion(outputs, x_orig)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Training Loss: {avg_train_loss:.4f}")


Avaluació i visualitzacio

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model.eval()
x_rot, x_orig = next(iter(test_loader))
x_rot, x_orig = x_rot.to(device), x_orig.to(device)

with torch.no_grad():
    recon = model(x_rot)

def show_images(rotated, original, reconstructed, n=5):
    plt.figure(figsize=(12,6))
    for i in range(n):
        plt.subplot(3, n, i+1)
        plt.imshow(np.transpose(rotated[i].cpu().numpy(), (1,2,0)))
        plt.title("Rotated")
        plt.axis('off')

        plt.subplot(3, n, i+1+n)
        plt.imshow(np.transpose(original[i].cpu().numpy(), (1,2,0)))
        plt.title("Original")
        plt.axis('off')

        plt.subplot(3, n, i+1+2*n)
        plt.imshow(np.transpose(reconstructed[i].cpu().numpy(), (1,2,0)))
        plt.title("Reconstructed")
        plt.axis('off')
    plt.show()

show_images(x_rot, x_orig, recon)
